In [51]:
import re

import nltk
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import  WordNetLemmatizer

In [14]:
data = pd.read_csv("tripadvisor_hotel_reviews.csv")
data.head()

,Review,Rating
0,nice hotel expensive parking got good deal sta...,4
1,ok nothing special charge diamond member hilto...,2
2,nice rooms not 4* experience hotel monaco seat...,3
3,"unique, great stay, wonderful time hotel monac...",5
4,"great stay great stay, went seahawk game aweso...",5


In [19]:
data['review_lower'] = data['Review'].str.lower()
data.head()

,Review,Rating,review_lower
0,nice hotel expensive parking got good deal sta...,4,nice hotel expensive parking got good deal sta...
1,ok nothing special charge diamond member hilto...,2,ok nothing special charge diamond member hilto...
2,nice rooms not 4* experience hotel monaco seat...,3,nice rooms not 4* experience hotel monaco seat...
3,"unique, great stay, wonderful time hotel monac...",5,"unique, great stay, wonderful time hotel monac..."
4,"great stay great stay, went seahawk game aweso...",5,"great stay great stay, went seahawk game aweso..."


In [21]:
en_stopwords = stopwords.words('english')
en_stopwords.remove('not')

In [24]:
data['review_no_stopwords'] = data['review_lower'].apply(
    lambda x: ' '.join([word for word in x.split() if word not in en_stopwords])
)
data['review_no_stopwords'][0]

'nice hotel expensive parking got good deal stay hotel anniversary, arrived late evening took advice previous reviews valet parking, check quick easy, little disappointed non-existent view room room clean nice size, bed comfortable woke stiff neck high pillows, not soundproof like heard music room night morning loud bangs doors opening closing hear people talking hallway, maybe noisy neighbors, aveda bath products nice, not goldfish stay nice touch taken advantage staying longer, location great walking distance shopping, overall nice experience pay 40 parking night,'

In [34]:
data['review_no_punt'] = data.apply(
    lambda x: re.sub(r"[*]"," star",x['review_no_stopwords']),axis=1
)

In [38]:
data['review_no_punt'] = data.apply(
    lambda x: re.sub(r"[^\w\s]","",x['review_no_punt']),axis=1
)

In [46]:
data['tokenized'] = data.apply(
    lambda x: word_tokenize(x['review_no_punt']),axis=1
)
data.head()

,Review,Rating,review_lower,review_no_stopwords,review_no_punt,tokenized
0,nice hotel expensive parking got good deal sta...,4,nice hotel expensive parking got good deal sta...,nice hotel expensive parking got good deal sta...,nice hotel expensive parking got good deal sta...,"[nice, hotel, expensive, parking, got, good, d..."
1,ok nothing special charge diamond member hilto...,2,ok nothing special charge diamond member hilto...,ok nothing special charge diamond member hilto...,ok nothing special charge diamond member hilto...,"[ok, nothing, special, charge, diamond, member..."
2,nice rooms not 4* experience hotel monaco seat...,3,nice rooms not 4* experience hotel monaco seat...,nice rooms not 4* experience hotel monaco seat...,nice rooms not 4 star experience hotel monaco ...,"[nice, rooms, not, 4, star, experience, hotel,..."
3,"unique, great stay, wonderful time hotel monac...",5,"unique, great stay, wonderful time hotel monac...","unique, great stay, wonderful time hotel monac...",unique great stay wonderful time hotel monaco ...,"[unique, great, stay, wonderful, time, hotel, ..."
4,"great stay great stay, went seahawk game aweso...",5,"great stay great stay, went seahawk game aweso...","great stay great stay, went seahawk game aweso...",great stay great stay went seahawk game awesom...,"[great, stay, great, stay, went, seahawk, game..."


In [53]:
ps = PorterStemmer()
data['review_stemmed'] = data['tokenized'].apply(
    lambda tokens: [ps.stem(token) for token in tokens]
)

In [57]:
lemma = WordNetLemmatizer()
data['review_lemmatized'] = data['tokenized'].apply(
    lambda tokens: [lemma.lemmatize(token) for token in tokens]
)

In [59]:
data.head()

,Review,Rating,review_lower,review_no_stopwords,review_no_punt,tokenized,review_stemmed,review_lemmatized
0,nice hotel expensive parking got good deal sta...,4,nice hotel expensive parking got good deal sta...,nice hotel expensive parking got good deal sta...,nice hotel expensive parking got good deal sta...,"[nice, hotel, expensive, parking, got, good, d...","[nice, hotel, expens, park, got, good, deal, s...","[nice, hotel, expensive, parking, got, good, d..."
1,ok nothing special charge diamond member hilto...,2,ok nothing special charge diamond member hilto...,ok nothing special charge diamond member hilto...,ok nothing special charge diamond member hilto...,"[ok, nothing, special, charge, diamond, member...","[ok, noth, special, charg, diamond, member, hi...","[ok, nothing, special, charge, diamond, member..."
2,nice rooms not 4* experience hotel monaco seat...,3,nice rooms not 4* experience hotel monaco seat...,nice rooms not 4* experience hotel monaco seat...,nice rooms not 4 star experience hotel monaco ...,"[nice, rooms, not, 4, star, experience, hotel,...","[nice, room, not, 4, star, experi, hotel, mona...","[nice, room, not, 4, star, experience, hotel, ..."
3,"unique, great stay, wonderful time hotel monac...",5,"unique, great stay, wonderful time hotel monac...","unique, great stay, wonderful time hotel monac...",unique great stay wonderful time hotel monaco ...,"[unique, great, stay, wonderful, time, hotel, ...","[uniqu, great, stay, wonder, time, hotel, mona...","[unique, great, stay, wonderful, time, hotel, ..."
4,"great stay great stay, went seahawk game aweso...",5,"great stay great stay, went seahawk game aweso...","great stay great stay, went seahawk game aweso...",great stay great stay went seahawk game awesom...,"[great, stay, great, stay, went, seahawk, game...","[great, stay, great, stay, went, seahawk, game...","[great, stay, great, stay, went, seahawk, game..."


In [61]:
tokens_clean = sum(data['review_lemmatized'],[])

In [63]:
unigram = (pd.Series(nltk.ngrams(tokens_clean,1)).value_counts())
print(unigram)

(hotel,)             52858
(room,)              46315
(not,)               31525
(great,)             21088
(nt,)                18993
                     ...  
(victimizedthis,)        1
(tallers,)               1
(cornerone,)             1
(pocketthere,)           1
(swears,)                1
Name: count, Length: 77404, dtype: int64


In [64]:
bigram = (pd.Series(nltk.ngrams(tokens_clean,2)).value_counts())
print(bigram)

(great, location)        2143
(staff, friendly)        2042
(ca, nt)                 1809
(room, clean)            1683
(punta, cana)            1682
                         ... 
(small, passable)           1
(passable, night)           1
(leave, following)          1
(recommend, extended)       1
(unless, tight)             1
Name: count, Length: 952527, dtype: int64
